# Phase 10 Main — Cross-task Spectral Signatures of Grounding Faithfulness

**Goal**: Establish that spectral features of token-entropy `H(n)` predict **statement-level grounding faithfulness** in long-context RAG, across **multiple models** and **multiple task types**, with characteristic **per-task spectral fingerprints** captured by the Nadler weight vector.

**Setup**:
- **4 models** × **4 datasets** = **16 cells**
  - Models: Qwen2.5-7B (within-family scale), Mistral-Small-24B (third family), Qwen2.5-72B-AWQ (flagship), Llama-3.3-70B-BNB (cross-family 70B)
  - Datasets: L-CiteEval HotpotQA, NaturalQuestions, 2WikiMultihopQA, NarrativeQA — same evaluator, four task styles
- **N=500 raw samples per cell** → ~600-1000 valid statements per cell
- **Total**: ~8,000 inference calls + ~8,000 Semantic-Entropy MC calls. Budget ~200 hours of A100 across multiple Colab sessions.

**What this notebook produces**:
1. A 4×4 AUC heatmap (the headline generalization claim)
2. A 16×13 Nadler weight matrix heatmap — **per-task spectral fingerprints**
3. Hierarchical clustering of weight vectors — does spectral signature cluster by model or by dataset?
4. 16-panel fusion-score distribution grid (grounded vs ungrounded per cell)
5. Length-controlled analysis per cell — addresses the length-confound risk that emerged in the pilot (trace_length AUC=70.7% on Qwen-72B)
6. Semantic Entropy baseline on 50 samples per cell — the must-beat reference
7. Full gate evaluation (G1–G4) per cell + cross-cell aggregate

**What this notebook does NOT include** (separate notebooks):
- Bracha LTT conformal calibration layer → `Spectral_Analysis_Phase10_Conformal.ipynb` (future)
- Agentic multi-step trajectories (DeepHalluBench, ReAct) → `Spectral_Analysis_Phase10_Agentic.ipynb` (future)

**Reads with**: `Phase10_Pilot_Plan.md`, `Phase10_Long_Doc_RAG_Design.md`, `research_phase10_rag/*.json`, `Research_Directions.md`.

## Section 1 — Setup

In [ ]:
# Cell 1 — Clone repo + install deps + imports + freeze pyarrow
import os, sys, shutil

# Set BEFORE any torch import — allocator uses expandable physical segments,
# so memory freed after one model can be reclaimed by a later/larger load.
# Materially raises the odds Llama-70B will fit after Mistral-24B is unloaded.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Persist HF cache to Drive — avoids re-downloading 36 GB AWQ on every restart.
# Set BEFORE any HF import.
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'

REPO_DIR = '/content/hallucination_detection'
if os.path.exists(REPO_DIR) and not os.path.exists(os.path.join(REPO_DIR, 'spectral_utils')):
    shutil.rmtree(REPO_DIR)
if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b master https://github.com/omrisegev/hallucination_detection.git {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull -q')
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# autoawq is safe here. gptqmodel is NOT — it rewrites numpy/pyarrow .so files
# during install and corrupts them mid-session. gptqmodel is installed later
# in the Qwen-72B-AWQ driver cell (see CLAUDE.md "Model loading rules").
os.system('pip install -q "transformers>=4.40" accelerate datasets bitsandbytes autoawq scipy seaborn')

from spectral_utils import (
    load_model, generate_full, free_memory,
    extract_all_features, sw_var_peak_adaptive,
    segment_by_citations, FEAT_NAMES,
    load_cache, save_cache,
    zscore, boot_auc, nadler_fuse, simple_average_fusion, best_nadler_on,
    load_lciteeval, lciteeval_prompt, lciteeval_grounding_label,
    lite_semantic_entropy_for_statement,
)

# Force-load datasets so pyarrow is frozen in memory before any later gptqmodel install.
import datasets  # noqa: F401

print('spectral_utils imported OK')

spectral_utils imported OK


In [ ]:
# Cell 2 — Master config
import torch, numpy as np

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

TEMP        = 1.0
MAX_NEW     = 1024
N_SAMPLES   = 500     # per (model, dataset) cell
SE_SAMPLES  = 50      # subset for the expensive Semantic-Entropy baseline
SE_K        = 10      # MC samples per statement for SE

# 4 models. Each entry is (key, model_id, load_kwargs, est_seconds_per_sample).
# load_kwargs is passed straight to load_model (which auto-detects AWQ).
MODELS = [
    ('qwen7b',     'Qwen/Qwen2.5-7B-Instruct',          {'quantize_4bit': False},  15),
    ('mistral24b', 'mistralai/Mistral-Small-24B-Instruct-2501', {'quantize_4bit': False}, 50),
    ('qwen72b',    'Qwen/Qwen2.5-72B-Instruct-AWQ',     {'quantize_4bit': False}, 110),  # AWQ → needs gptqmodel install before load
    ('llama70b',   'meta-llama/Llama-3.3-70B-Instruct', {'quantize_4bit': True},  140),  # BNB 4-bit on A100 80GB
]

# 4 L-CiteEval sub-tasks — same evaluator, different task styles.
DATASETS = ['hotpotqa', 'natural_questions', '2wikimultihopqa', 'narrativeqa']

BASE_DIR  = '/content/drive/MyDrive/hallucination_detection/cache/phase10_main'
RAW_DIR   = f'{BASE_DIR}/raw'         # one .pkl per (model, dataset) — raw generations + entropies
FEAT_DIR  = f'{BASE_DIR}/features'    # one .pkl per (model, dataset) — extracted spectral features + labels
RES_DIR   = f'{BASE_DIR}/results'     # final aggregate results
PLOT_DIR  = f'{BASE_DIR}/plots'

print(f'Models:   {[m[0] for m in MODELS]}')
print(f'Datasets: {DATASETS}')
print(f'Plan:     {len(MODELS)} \u00d7 {len(DATASETS)} = {len(MODELS)*len(DATASETS)} cells, N={N_SAMPLES}/cell')

est_hours = sum(N_SAMPLES * s for _,_,_,s in MODELS) * len(DATASETS) / 3600
print(f'Compute estimate (inference only): {est_hours:.1f} hours')

Models:   ['qwen7b', 'mistral24b', 'qwen72b', 'llama70b']
Datasets: ['hotpotqa', 'natural_questions', '2wikimultihopqa', 'narrativeqa']
Plan:     4 × 4 = 16 cells, N=500/cell
Compute estimate (inference only): 175.0 hours


In [ ]:
# Cell 3 — Mount Drive + create cache dirs
from google.colab import drive
drive.mount('/content/drive')

for d in (BASE_DIR, RAW_DIR, FEAT_DIR, RES_DIR, PLOT_DIR):
    os.makedirs(d, exist_ok=True)
print(f'Cache root: {BASE_DIR}')

Mounted at /content/drive
Cache root: /content/drive/MyDrive/hallucination_detection/cache/phase10_main


In [ ]:
# Cell 3b — HF cache diagnostic: is the AWQ download persisting to Drive?
#
# Observed: 17.8 GB AWQ re-downloads on each runtime restart even though
# HF_HOME points to Drive. Likely cause: HuggingFace's hub cache uses
# symlinks for blob dedup, and Google Drive doesn't support real symlinks.
# This cell reports what HF actually sees so we can pick the right fix.

import huggingface_hub
from huggingface_hub import constants as _hfc

print('Env vars:')
print(f'  HF_HOME       = {os.environ.get("HF_HOME", "(unset)")}')
print(f'  HF_HUB_CACHE  = {os.environ.get("HF_HUB_CACHE", "(unset)")}')
print(f'  TRANSFORMERS_CACHE = {os.environ.get("TRANSFORMERS_CACHE", "(unset)")}')
print()
print('huggingface_hub.constants (frozen at first import):')
print(f'  HF_HOME       = {_hfc.HF_HOME}')
print(f'  HF_HUB_CACHE  = {_hfc.HF_HUB_CACHE}')
print()
print('Filesystem check:')
for _path in (_hfc.HF_HOME, _hfc.HF_HUB_CACHE, '/content/drive/MyDrive/hf_cache'):
    _exists = os.path.exists(_path)
    _is_dir = os.path.isdir(_path) if _exists else False
    _size_gb = 0.0
    if _is_dir:
        for _root, _, _files in os.walk(_path):
            for _f in _files:
                try: _size_gb += os.path.getsize(os.path.join(_root, _f))
                except OSError: pass
        _size_gb /= 1e9
    print(f'  {_path}: exists={_exists} is_dir={_is_dir} size={_size_gb:.2f} GB')

# Check if a Qwen2.5-72B-AWQ snapshot already lives on Drive
_qwen_path = os.path.join(_hfc.HF_HUB_CACHE, 'models--Qwen--Qwen2.5-72B-Instruct-AWQ')
print(f'\nQwen-72B-AWQ snapshot dir: {_qwen_path}')
print(f'  exists: {os.path.exists(_qwen_path)}')
if os.path.exists(_qwen_path):
    for _root, _dirs, _files in os.walk(_qwen_path):
        for _f in _files[:5]:
            _p = os.path.join(_root, _f)
            _is_link = os.path.islink(_p)
            _size = os.path.getsize(_p) if not _is_link else 0
            print(f'    {_p}: islink={_is_link} size={_size}')

Env vars:
  HF_HOME       = /content/drive/MyDrive/hf_cache
  HF_HUB_CACHE  = (unset)
  TRANSFORMERS_CACHE = (unset)

huggingface_hub.constants (frozen at first import):
  HF_HOME       = /content/drive/MyDrive/hf_cache
  HF_HUB_CACHE  = /content/drive/MyDrive/hf_cache/hub

Filesystem check:
  /content/drive/MyDrive/hf_cache: exists=True is_dir=True size=431.29 GB
  /content/drive/MyDrive/hf_cache/hub: exists=True is_dir=True size=431.14 GB
  /content/drive/MyDrive/hf_cache: exists=True is_dir=True size=431.29 GB

Qwen-72B-AWQ snapshot dir: /content/drive/MyDrive/hf_cache/hub/models--Qwen--Qwen2.5-72B-Instruct-AWQ
  exists: True
    /content/drive/MyDrive/hf_cache/hub/models--Qwen--Qwen2.5-72B-Instruct-AWQ/blobs/d268bce9d32327f1be2638d6d00e2fe6aa853bb4: islink=False size=841
    /content/drive/MyDrive/hf_cache/hub/models--Qwen--Qwen2.5-72B-Instruct-AWQ/blobs/07bfe0640cb5a0037f9322287fbfc682806cf672: islink=False size=7305
    /content/drive/MyDrive/hf_cache/hub/models--Qwen--Qwen2.5-72

In [ ]:
# Cell 3c — Flat-dir downloader for large models (workaround for Drive's broken symlinks)
#
# Drive's FUSE doesn't support real symlinks, so HF's standard hub cache structure
# (blobs/ as real files, snapshots/<rev>/ as symlinks) ends up with 0-byte broken
# symlinks. Each session, HF can't resolve the snapshot dir and re-downloads.
#
# Fix: bypass the cache. snapshot_download(local_dir=...) downloads to a flat
# directory of real files (no symlinks). Then pass that path to from_pretrained
# instead of the Hub repo ID. load_model still detects AWQ from "awq" in the path.

from huggingface_hub import snapshot_download

FLAT_CACHE = '/content/drive/MyDrive/hf_cache_flat'
os.makedirs(FLAT_CACHE, exist_ok=True)

def ensure_flat_dir(repo_id, token=None):
    """
    Download a repo to a flat dir on Drive (real files, no symlinks). Idempotent —
    re-running this skips the download once `config.json` is present.
    Returns the local path; pass it to load_model() instead of the Hub repo ID.
    """
    safe_name = repo_id.replace('/', '__')
    local_dir = os.path.join(FLAT_CACHE, safe_name)
    sentinel = os.path.join(local_dir, 'config.json')
    if os.path.exists(sentinel):
        size_gb = sum(os.path.getsize(os.path.join(r, f))
                      for r, _, fs in os.walk(local_dir) for f in fs) / 1e9
        print(f'  [{repo_id}] already on Drive ({size_gb:.1f} GB) at {local_dir}')
        return local_dir
    print(f'  [{repo_id}] downloading to flat dir on Drive (one-time)...')
    kwargs = dict(repo_id=repo_id, local_dir=local_dir, token=token)
    try:
        snapshot_download(**kwargs, local_dir_use_symlinks=False)
    except TypeError:
        # Newer huggingface_hub (>=0.23) removed local_dir_use_symlinks;
        # local_dir already defaults to copies in those versions.
        snapshot_download(**kwargs)
    print(f'  [{repo_id}] flat-dir download done.')
    return local_dir

print('ensure_flat_dir helper ready.')

ensure_flat_dir helper ready.


In [ ]:
# Cell 4 — Pre-load all 4 datasets (cached via HF datasets cache on Drive)
import pickle

DATA = {}  # name -> list of normalized rows
for ds_name in DATASETS:
    cached = os.path.join(BASE_DIR, f'data_{ds_name}_n{N_SAMPLES}.pkl')
    if os.path.exists(cached):
        with open(cached, 'rb') as f:
            DATA[ds_name] = pickle.load(f)
        print(f'  [{ds_name}] loaded {len(DATA[ds_name])} from cache')
    else:
        DATA[ds_name] = load_lciteeval(ds_name, n_samples=N_SAMPLES)
        with open(cached, 'wb') as f:
            pickle.dump(DATA[ds_name], f)
        print(f'  [{ds_name}] downloaded {len(DATA[ds_name])} and cached')

  [hotpotqa] loaded 240 from cache
  [natural_questions] loaded 160 from cache
  [2wikimultihopqa] loaded 240 from cache
  [narrativeqa] loaded 240 from cache


In [ ]:
# Cell 5 — Spot-check one sample per dataset
for ds_name in DATASETS:
    sample = DATA[ds_name][0]
    print(f'\n=== {ds_name} ===')
    print(f'Q: {sample["question"][:160]}...')
    print(f'Docs: {len(sample["docs"])}  Answers: {sample["answers"]}')
    print(f'Has supporting_facts: {bool(sample["raw_row"].get("supporting_facts"))}')
    if sample['docs']:
        print(f'First doc title: {sample["docs"][0]["title"][:80]}')


=== hotpotqa ===
Q: Where did Henri Christophe and other slaves hold an uprising from 1791 to 1804 that led to the founding of a state which was both free from slavery and ruled by...
Docs: 59  Answers: ['Saint-Domingue']
Has supporting_facts: False
First doc title: Passage 1

=== natural_questions ===
Q: what tectonic setting is responsible for the folded mountains of pennsylvania and the high himalaya?...
Docs: 88  Answers: [['a convergent plate boundary']]
Has supporting_facts: False
First doc title: <!DOCTYPE html>

=== 2wikimultihopqa ===
Q: Who died first, Erich Haenisch or William Pooley?...
Docs: 60  Answers: ['William Pooley']
Has supporting_facts: False
First doc title: Passage 1

=== narrativeqa ===
Q: What were the name of the two mice?...
Docs: 23  Answers: [['Tom Thumb and Hunca Munca', 'Tom Thumb and Hunca Munca.']]
Has supporting_facts: False
First doc title: ï»¿The Project Gutenberg EBook of The Tale of Two Bad Mice, by Beatrix Potter


## Section 2 — Inference (16 cells)

Each driver cell loads ONE model and runs it across ALL 4 datasets, then unloads it. Per-(model, dataset) checkpointing inside `run_inference_for_cell` saves every 25 samples — safe to interrupt and resume.

**Order**: smallest → largest, so you can interrupt and still have results from cheaper models if compute runs out.

**Important**: After each driver cell, the model is unloaded. The next driver cell loads its own model from scratch. This is by design — keeps GPU memory clean between models.

In [ ]:
# Cell 6 — Workhorse inference function
from tqdm.auto import tqdm

def raw_path(model_key, ds_name):
    return os.path.join(RAW_DIR, f'{model_key}__{ds_name}.pkl')

def run_inference_for_cell(mdl, tok, model_key, ds_name, n_samples=N_SAMPLES,
                            checkpoint_every=25):
    """
    Run inference for one (model, dataset) cell.
    Resumes from the .pkl checkpoint if present.
    Caps at min(n_samples, len(dataset)) — L-CiteEval test splits are smaller
    than 500 (HotpotQA=240, NaturalQuestions=160, 2WikiMHQA=240, NarrativeQA=240).
    """
    path = raw_path(model_key, ds_name)
    results = []
    if os.path.exists(path):
        with open(path, 'rb') as f:
            results = pickle.load(f)
        print(f'    [{model_key}/{ds_name}] resumed from {len(results)} samples')

    # Hard cap at actual dataset size — avoids index errors when n_samples > dataset length
    actual_n = min(n_samples, len(DATA[ds_name]))
    start = len(results)
    if start >= actual_n:
        print(f'    [{model_key}/{ds_name}] already complete ({len(results)}/{actual_n})')
        return

    rows = DATA[ds_name]
    for i in tqdm(range(start, actual_n), desc=f'{model_key}/{ds_name}'):
        try:
            res = generate_full(mdl, tok, lciteeval_prompt(rows[i]),
                                T=TEMP, max_new=MAX_NEW)
            results.append({'idx': i, 'row': rows[i], 'output': res})
        except Exception as ex:
            print(f'    error idx={i}: {ex}')
            continue
        if (i + 1) % checkpoint_every == 0:
            with open(path, 'wb') as f:
                pickle.dump(results, f)
    with open(path, 'wb') as f:
        pickle.dump(results, f)
    print(f'    [{model_key}/{ds_name}] complete: {len(results)}/{actual_n} saved')

def status_for_model(model_key):
    """Print per-dataset progress for one model."""
    print(f'  status [{model_key}]:')
    for ds in DATASETS:
        p = raw_path(model_key, ds)
        n = len(pickle.load(open(p, 'rb'))) if os.path.exists(p) else 0
        actual = min(N_SAMPLES, len(DATA[ds])) if ds in DATA else N_SAMPLES
        print(f'    {ds:<22s} {n}/{actual}')

print('Helpers ready.')

Helpers ready.


### Driver — Qwen2.5-7B-Instruct (smallest, fastest, within-family scale ablation)

In [ ]:
# Cell 7 — Qwen-7B on all 4 datasets
MODEL_KEY, MODEL_ID, KW, _ = MODELS[0]
mdl, tok = load_model(MODEL_ID, **KW)
print(f'GPU: {torch.cuda.memory_allocated()/1e9:.1f} / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

for ds in DATASETS:
    run_inference_for_cell(mdl, tok, MODEL_KEY, ds)

del mdl, tok; free_memory()
status_for_model(MODEL_KEY)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

    error idx=160: list index out of range
    error idx=161: list index out of range
    error idx=162: list index out of range
    error idx=163: list index out of range
    error idx=164: list index out of range
    error idx=165: list index out of range
    error idx=166: list index out of range
    error idx=167: list index out of range
    error idx=168: list index out of range
    error idx=169: list index out of range
    error idx=170: list index out of range
    error idx=171: list index out of range
    error idx=172: list index out of range
    error idx=173: list index out of range
    error idx=174: list index out of range
    error idx=175: list index out of range
    error idx=176: list index out of range
    error idx=177: list index out of range
    error idx=178: list index out of range
    error idx=179: list index out of range
    error idx=180: list index out of range
    error idx=181: list index out of range
    error idx=182: list index out of range
    error i

qwen7b/2wikimultihopqa:   0%|          | 0/500 [00:00<?, ?it/s]

    error idx=240: list index out of range
    error idx=241: list index out of range
    error idx=242: list index out of range
    error idx=243: list index out of range
    error idx=244: list index out of range
    error idx=245: list index out of range
    error idx=246: list index out of range
    error idx=247: list index out of range
    error idx=248: list index out of range
    error idx=249: list index out of range
    error idx=250: list index out of range
    error idx=251: list index out of range
    error idx=252: list index out of range
    error idx=253: list index out of range
    error idx=254: list index out of range
    error idx=255: list index out of range
    error idx=256: list index out of range
    error idx=257: list index out of range
    error idx=258: list index out of range
    error idx=259: list index out of range
    error idx=260: list index out of range
    error idx=261: list index out of range
    error idx=262: list index out of range
    error i

qwen7b/narrativeqa:   0%|          | 0/500 [00:00<?, ?it/s]

    error idx=240: list index out of range
    error idx=241: list index out of range
    error idx=242: list index out of range
    error idx=243: list index out of range
    error idx=244: list index out of range
    error idx=245: list index out of range
    error idx=246: list index out of range
    error idx=247: list index out of range
    error idx=248: list index out of range
    error idx=249: list index out of range
    error idx=250: list index out of range
    error idx=251: list index out of range
    error idx=252: list index out of range
    error idx=253: list index out of range
    error idx=254: list index out of range
    error idx=255: list index out of range
    error idx=256: list index out of range
    error idx=257: list index out of range
    error idx=258: list index out of range
    error idx=259: list index out of range
    error idx=260: list index out of range
    error idx=261: list index out of range
    error idx=262: list index out of range
    error i

### Driver — Mistral-Small-24B-Instruct (third family)

In [ ]:
# Cell 8 — Mistral-24B on all 4 datasets
MODEL_KEY, MODEL_ID, KW, _ = MODELS[1]
mdl, tok = load_model(MODEL_ID, **KW)
print(f'GPU: {torch.cuda.memory_allocated()/1e9:.1f} / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

for ds in DATASETS:
    run_inference_for_cell(mdl, tok, MODEL_KEY, ds)

del mdl, tok; free_memory()
status_for_model(MODEL_KEY)

config.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The tokenizer you are loading from 'mistralai/Mistral-Small-24B-Instruct-2501' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


...

    error idx=160: list index out of range
    error idx=161: list index out of range
    error idx=162: list index out of range
    error idx=163: list index out of range
    error idx=164: list index out of range
    error idx=165: list index out of range
    error idx=166: list index out of range
    error idx=167: list index out of range
    error idx=168: list index out of range
    error idx=169: list index out of range
    error idx=170: list index out of range
    error idx=171: list index out of range
    error idx=172: list index out of range
    error idx=173: list index out of range
    error idx=174: list index out of range
    error idx=175: list index out of range
    error idx=176: list index out of range
    error idx=177: list index out of range
    error idx=178: list index out of range
    error idx=179: list index out of range
    error idx=180: list index out of range
    error idx=181: list index out of range
    error idx=182: list index out of range
    error i

mistral24b/2wikimultihopqa:   0%|          | 0/500 [00:00<?, ?it/s]

    error idx=240: list index out of range
    error idx=241: list index out of range
    error idx=242: list index out of range
    error idx=243: list index out of range
    error idx=244: list index out of range
    error idx=245: list index out of range
    error idx=246: list index out of range
    error idx=247: list index out of range
    error idx=248: list index out of range
    error idx=249: list index out of range
    error idx=250: list index out of range
    error idx=251: list index out of range
    error idx=252: list index out of range
    error idx=253: list index out of range
    error idx=254: list index out of range
    error idx=255: list index out of range
    error idx=256: list index out of range
    error idx=257: list index out of range
    error idx=258: list index out of range
    error idx=259: list index out of range
    error idx=260: list index out of range
    error idx=261: list index out of range
    error idx=262: list index out of range
    error i

mistral24b/narrativeqa:   0%|          | 0/500 [00:00<?, ?it/s]

    error idx=240: list index out of range
    error idx=241: list index out of range
    error idx=242: list index out of range
    error idx=243: list index out of range
    error idx=244: list index out of range
    error idx=245: list index out of range
    error idx=246: list index out of range
    error idx=247: list index out of range
    error idx=248: list index out of range
    error idx=249: list index out of range
    error idx=250: list index out of range
    error idx=251: list index out of range
    error idx=252: list index out of range
    error idx=253: list index out of range
    error idx=254: list index out of range
    error idx=255: list index out of range
    error idx=256: list index out of range
    error idx=257: list index out of range
    error idx=258: list index out of range
    error idx=259: list index out of range
    error idx=260: list index out of range
    error idx=261: list index out of range
    error idx=262: list index out of range
    error i

### Driver — Qwen2.5-72B-Instruct-AWQ (flagship)

AWQ requires the `gptqmodel` Marlin kernel. Install it **inside the driver cell**, after all earlier cells imported pyarrow/datasets — gptqmodel rewrites those .so files on disk and would otherwise break the running kernel.

In [ ]:
# Cell 9 — Qwen-72B-AWQ on all 4 datasets
#
# pcre stub rationale: gptqmodel + defuser + tokenicer all import `pcre`, which
# is pypcre on PyPI (C extension over libpcre2, no Py3.12 wheel). Stub it with
# stdlib re. Observed call sites: pcre.compile/.sub, pcre.Pattern/Match,
# pcre.Flag.{CASELESS,MULTILINE,EXTENDED}.
# Dep install: --no-deps on gptqmodel avoids transformers rewrites; install
# device-smi/tokenicer/defuser/logbar/ninja as runtime deps (ninja JIT-builds
# gptqmodel's Marlin fp16 CUDA kernel at first load; without it AWQ errors at
# the very last step of model materialization).
# Model load: use ensure_flat_dir() to bypass HF's broken-on-Drive symlink
# cache (see Cell 3c).

# (1) Stub pcre with stdlib re BEFORE any gptqmodel import.
import re as _re, types as _types
_pcre = _types.ModuleType('pcre')
for _fn in ('compile', 'match', 'search', 'findall', 'sub', 'split', 'fullmatch'):
    setattr(_pcre, _fn, getattr(_re, _fn))
_pcre.Pattern = _re.Pattern
_pcre.Match   = _re.Match
_pcre.error   = _re.error
_re_flag_map = {
    'IGNORECASE': _re.IGNORECASE,      'CASELESS':       _re.IGNORECASE,
    'MULTILINE':  _re.MULTILINE,
    'DOTALL':     _re.DOTALL,
    'VERBOSE':    _re.VERBOSE,         'EXTENDED':       _re.VERBOSE,
    'UNICODE':    _re.UNICODE,         'UTF':            _re.UNICODE,
                                       'UTF8':           _re.UNICODE,
                                       'UCP':            _re.UNICODE,
    'ASCII':      _re.ASCII,
    'ANCHORED':         0, 'UNGREEDY':           0, 'DOLLAR_ENDONLY':   0,
    'NO_AUTO_CAPTURE':  0, 'NO_AUTO_POSSESS':    0, 'NEVER_BACKSLASH_C': 0,
    'NEVER_UCP':        0, 'NEVER_UTF':          0, 'NO_DOTSTAR_ANCHOR': 0,
    'NO_START_OPTIMIZE': 0, 'USE_OFFSET_LIMIT':  0, 'ENDANCHORED':      0,
}
for _name, _val in _re_flag_map.items():
    setattr(_pcre, _name, _val)
_pcre.Flag = _types.SimpleNamespace(**_re_flag_map)
sys.modules['pcre'] = _pcre
print('pcre stubbed with stdlib re')

# (2) Install gptqmodel runtime deps that --no-deps will skip.
#     ninja is required at model-load time to JIT-build the Marlin fp16 CUDA
#     kernel (gptqmodel/nn_modules/qlinear/marlin_awq.py:post_init).
os.system('pip install -q --no-deps device-smi tokenicer defuser')
os.system('pip install -q logbar ninja')

# (3) Install gptqmodel itself without auto-pulling deps.
os.system('pip install -q --no-deps gptqmodel')

# (4) Resolve model to a flat dir on Drive (no broken symlinks), then load.
MODEL_KEY, MODEL_ID, KW, _ = MODELS[2]
LOCAL_PATH = ensure_flat_dir(MODEL_ID)
mdl, tok = load_model(LOCAL_PATH, **KW)
print(f'GPU: {torch.cuda.memory_allocated()/1e9:.1f} / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

for ds in DATASETS:
    run_inference_for_cell(mdl, tok, MODEL_KEY, ds)

del mdl, tok; free_memory()
status_for_model(MODEL_KEY)

pcre stubbed with stdlib re
  [Qwen/Qwen2.5-72B-Instruct-AWQ] downloading to flat dir on Drive (one-time)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 21 files:   0%|          | 0/21 [00:00<?, ?it/s]

  [Qwen/Qwen2.5-72B-Instruct-AWQ] flat-dir download done.


`torch.bfloat16` is not supported for AWQ CUDA/XPU kernels yet. Casting to `torch.float16`.


...

    [qwen72b/natural_questions] complete: 160/160 saved


qwen72b/2wikimultihopqa:   0%|          | 0/240 [00:00<?, ?it/s]

    [qwen72b/2wikimultihopqa] complete: 240/240 saved


qwen72b/narrativeqa:   0%|          | 0/240 [00:00<?, ?it/s]

    [qwen72b/narrativeqa] complete: 240/240 saved
  status [qwen72b]:
    hotpotqa               240/240
    natural_questions      160/160
    2wikimultihopqa        240/240
    narrativeqa            240/240


### Driver — Llama-3.3-70B-Instruct (BNB 4-bit, cross-family 70B)

In [ ]:
# Cell 10 — Llama-70B on all 4 datasets
#
# Loading constraints — read before running:
#
# 1. Llama-3.3-70B is a gated repo. Needs HF_TOKEN in Colab Secrets.
# 2. 72B-class BNB needs device_map={'':0}. With device_map='auto', BNB sees
#    the pre-quant FP16 size (~145 GB) and dispatches layers to CPU → BNB
#    aborts with "modules dispatched on CPU".
# 3. NEVER pass torch_dtype alongside quantization_config — bitsandbytes owns
#    the dtype, and passing it bypasses BNB and loads full FP16 → OOM.
# 4. Peak GPU during 4-bit quantization is ~80 GB. After any smaller model has
#    been loaded and freed, fragmentation causes OOM. The freshness guard below
#    refuses the load when the runtime is not fresh.
# 5. Weights stored as a flat dir on Drive (see Cell 3c) — no broken symlinks.

# Best-effort cleanup before the freshness check — recovers what can be recovered.
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

_peak_gb = torch.cuda.max_memory_allocated() / 1e9
_alloc_gb = torch.cuda.memory_allocated() / 1e9
print(f'GPU before load — alloc: {_alloc_gb:.1f} GB | peak this runtime: {_peak_gb:.1f} GB')

if _peak_gb > 5.0:
    print('')
    print('=' * 70)
    print('⚠️  Llama-70B load REFUSED — this runtime has already used GPU memory.')
    print(f'   Peak in this runtime: {_peak_gb:.1f} GB (threshold: 5 GB).')
    print('   70B BNB 4-bit needs ~80 GB peak during quantization. After any')
    print('   other model has been loaded, fragmentation reliably causes OOM.')
    print('')
    print('   To run Llama-70B successfully:')
    print('     1. Runtime → Restart runtime  (fresh CUDA context)')
    print('     2. Run Cells 1–6  (config + helpers, no model load)')
    print('     3. SKIP Cells 7, 8, 9   (smaller models)')
    print('     4. Run Cell 10 directly — checkpoints from prior runs are kept')
    print('=' * 70)
    status_for_model('llama70b')
else:
    from google.colab import userdata
    try:
        hf_token = userdata.get('HF_TOKEN')
    except Exception as ex:
        raise RuntimeError(
            'HF_TOKEN missing from Colab Secrets. Llama-3.3-70B is gated — '
            'add HF_TOKEN with read access to meta-llama/Llama-3.3-70B-Instruct.'
        ) from ex

    MODEL_KEY, MODEL_ID, KW, _ = MODELS[3]
    LOCAL_PATH = ensure_flat_dir(MODEL_ID, token=hf_token)

    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    tok = AutoTokenizer.from_pretrained(LOCAL_PATH)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
    mdl = AutoModelForCausalLM.from_pretrained(
        LOCAL_PATH,
        device_map={'': 0},
        quantization_config=bnb,
        attn_implementation='eager',
        trust_remote_code=False,
    ).eval()
    print(f'Loaded {MODEL_ID} [BNB 4-bit, device_map=GPU0]')
    print(f'GPU: {torch.cuda.memory_allocated()/1e9:.1f} / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

    for ds in DATASETS:
        run_inference_for_cell(mdl, tok, MODEL_KEY, ds)

    del mdl, tok; free_memory()
    status_for_model(MODEL_KEY)

GPU before load — alloc: 0.0 GB | peak this runtime: 0.0 GB
  [meta-llama/Llama-3.3-70B-Instruct] already on Drive (282.3 GB) at /content/drive/MyDrive/hf_cache_flat/meta-llama__Llama-3.3-70B-Instruct


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 448.00 MiB. GPU 0 has a total capacity of 79.25 GiB of which 440.81 MiB is free. Including non-PyTorch memory, this process has 78.81 GiB memory in use. Of the allocated memory 78.30 GiB is allocated by PyTorch, and 25.43 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Section 3 — Per-cell feature extraction

In [ ]:
# Cell 11 — Extract spectral features per (model, dataset) cell
#
# Skips cells where the raw .pkl is missing (so this can run while inference
# is still in progress on some cells).

def feat_path(model_key, ds_name):
    return os.path.join(FEAT_DIR, f'{model_key}__{ds_name}.pkl')

def extract_for_cell(model_key, ds_name):
    rp = raw_path(model_key, ds_name)
    if not os.path.exists(rp):
        return None
    with open(rp, 'rb') as f:
        results = pickle.load(f)

    valid_statements, all_feats, all_labels = [], [], []
    n_no_cite = n_too_short = 0

    for r in results:
        ents    = r['output']['token_entropies']
        offsets = list(r['output']['token_offsets'])
        n = min(len(ents), len(offsets))
        ents, offsets = ents[:n], offsets[:n]
        segs = segment_by_citations(r['output']['full_text'], offsets)
        if not segs:
            n_no_cite += 1
            continue
        for seg in segs:
            seg_ents = ents[seg['token_start']:seg['token_end']]
            if len(seg_ents) < 10:
                n_too_short += 1
                continue
            f_ = extract_all_features(seg_ents)
            if f_ is None:
                n_too_short += 1
                continue
            f_['sw_var_peak_adaptive'] = sw_var_peak_adaptive(seg_ents)
            valid_statements.append({
                'sample_idx':   r['idx'],
                'text':         seg['text'],
                'citation_ids': seg['citation_ids'],
                'token_start_in_response': seg['token_start'],
                'ents':         seg_ents,
                'row':          r['row'],
                'full_text':    r['output']['full_text'],
            })
            all_feats.append(f_)
            all_labels.append(lciteeval_grounding_label(seg['citation_ids'], r['row']))

    out = {
        'model_key': model_key,
        'ds_name':   ds_name,
        'n_raw_samples':            len(results),
        'n_samples_with_citations': len(results) - n_no_cite,
        'citation_rate':            (len(results) - n_no_cite) / max(1, len(results)),
        'n_valid_statements':       len(valid_statements),
        'n_too_short':              n_too_short,
        'feat_keys':                list(all_feats[0].keys()) if all_feats else [],
        'features':                 all_feats,
        'labels':                   all_labels,
        'statements':               valid_statements,
    }
    with open(feat_path(model_key, ds_name), 'wb') as f:
        pickle.dump(out, f)
    return out

ALL_CELLS = {}
for mk, _, _, _ in MODELS:
    for ds in DATASETS:
        cell = extract_for_cell(mk, ds)
        if cell is None:
            print(f'  [{mk}/{ds}] missing raw — skipped')
        else:
            ALL_CELLS[(mk, ds)] = cell
            print(f'  [{mk}/{ds}] cite_rate={100*cell["citation_rate"]:.1f}%  '
                  f'valid_stmts={cell["n_valid_statements"]}  '
                  f'pos/neg={sum(cell["labels"])}/{len(cell["labels"])-sum(cell["labels"])}')

print(f'\nLoaded {len(ALL_CELLS)} of {len(MODELS)*len(DATASETS)} cells.')

  [qwen7b/hotpotqa] cite_rate=65.8%  valid_stmts=169  pos/neg=15/154
  [qwen7b/natural_questions] cite_rate=54.4%  valid_stmts=103  pos/neg=11/92
  [qwen7b/2wikimultihopqa] cite_rate=69.2%  valid_stmts=192  pos/neg=14/178
  [qwen7b/narrativeqa] cite_rate=82.9%  valid_stmts=232  pos/neg=27/205
  [mistral24b/hotpotqa] cite_rate=15.0%  valid_stmts=46  pos/neg=18/28
  [mistral24b/natural_questions] cite_rate=27.5%  valid_stmts=44  pos/neg=9/35
  [mistral24b/2wikimultihopqa] cite_rate=17.9%  valid_stmts=46  pos/neg=15/31
  [mistral24b/narrativeqa] cite_rate=55.8%  valid_stmts=237  pos/neg=33/204
  [qwen72b/hotpotqa] cite_rate=63.7%  valid_stmts=169  pos/neg=26/143
  [qwen72b/natural_questions] cite_rate=91.9%  valid_stmts=171  pos/neg=22/149
  [qwen72b/2wikimultihopqa] cite_rate=80.0%  valid_stmts=198  pos/neg=23/175
  [qwen72b/narrativeqa] cite_rate=97.5%  valid_stmts=344  pos/neg=36/308
  [llama70b/hotpotqa] missing raw — skipped
  [llama70b/natural_questions] missing raw — skipped
  [lla

In [ ]:
# Cell 12 — Pre-conditions (G0) per cell
import pandas as pd

g0_rows = []
for (mk, ds), c in ALL_CELLS.items():
    n_pos = sum(c['labels']); n_neg = len(c['labels']) - n_pos
    g0_rows.append({
        'model':          mk,
        'dataset':        ds,
        'cite_rate':      f'{100*c["citation_rate"]:.1f}%',
        'valid_stmts':    c['n_valid_statements'],
        'grounded':       n_pos,
        'ungrounded':     n_neg,
        'G0-A (cite\u226560%)':   '\u2713' if c['citation_rate'] >= 0.60 else '\u2717',
        'G0-B (n\u2265100)':      '\u2713' if c['n_valid_statements'] >= 100 else '\u2717',
        'G0-C (both\u226510)':    '\u2713' if (n_pos >= 10 and n_neg >= 10) else '\u2717',
    })
df_g0 = pd.DataFrame(g0_rows)
print('=== G0 Pre-conditions per cell ===')
print(df_g0.to_string(index=False))

=== G0 Pre-conditions per cell ===
     model           dataset cite_rate  valid_stmts  grounded  ungrounded G0-A (cite≥60%) G0-B (n≥100) G0-C (both≥10)
    qwen7b          hotpotqa     65.8%          169        15         154               ✓            ✓              ✓
    qwen7b natural_questions     54.4%          103        11          92               ✗            ✓              ✓
    qwen7b   2wikimultihopqa     69.2%          192        14         178               ✓            ✓              ✓
    qwen7b       narrativeqa     82.9%          232        27         205               ✓            ✓              ✓
mistral24b          hotpotqa     15.0%           46        18          28               ✗            ✗              ✓
mistral24b natural_questions     27.5%           44         9          35               ✗            ✗              ✗
mistral24b   2wikimultihopqa     17.9%           46        15          31               ✗            ✗              ✓
mistral24b       narr

## Section 4 — Per-cell analysis

In [ ]:
# Cell 13 — Individual feature AUC table per cell (16 × 13 matrix)
FEAT_COLS = None
auc_records = []
for (mk, ds), c in ALL_CELLS.items():
    if not c['features']:
        continue
    keys = list(c['features'][0].keys())
    FEAT_COLS = FEAT_COLS or keys
    labels_arr = np.array(c['labels'])
    row = {'model': mk, 'dataset': ds}
    for k in keys:
        scores = np.array([f[k] for f in c['features']])
        a, lo, hi = boot_auc(labels_arr, scores)
        if not np.isnan(a) and a < 0.5:
            a, lo, hi = boot_auc(labels_arr, -scores)
        row[k] = a
    auc_records.append(row)
df_auc = pd.DataFrame(auc_records)
print('=== Individual Feature AUCs (16 cells \u00d7 13 features) ===')
with pd.option_context('display.max_columns', None, 'display.width', 200,
                       'display.float_format', lambda x: f'{x:.3f}' if isinstance(x, float) else str(x)):
    print(df_auc.to_string(index=False))

=== Individual Feature AUCs (16 cells × 13 features) ===
     model           dataset   epr  trace_length  spectral_entropy  low_band_power  high_band_power  hl_ratio  dominant_freq  spectral_centroid  stft_max_high_power  stft_spectral_entropy  rpdi  sw_var_peak  sw_var_peak_adaptive
    qwen7b          hotpotqa 0.583         0.668             0.668           0.568            0.577     0.513          0.585              0.513                0.783                  0.609 0.614        0.568                 0.531
    qwen7b natural_questions 0.585         0.691             0.642           0.618            0.573     0.529          0.508              0.525                0.703                  0.601 0.512        0.626                 0.621
    qwen7b   2wikimultihopqa 0.571         0.618             0.646           0.704            0.537     0.618          0.613              0.555                0.624                  0.651 0.599        0.534                 0.541
    qwen7b       narrativeq

In [ ]:
# Cell 14 — Best Nadler subset + weights per cell
import itertools

NADLER_RES = {}  # (model, dataset) -> {auc, lo, hi, subset, weights, signs}
for (mk, ds), c in ALL_CELLS.items():
    if not c['features']:
        continue
    keys = list(c['features'][0].keys())
    feat_dict = {k: np.array([f[k] for f in c['features']]) for k in keys}
    labels_arr = np.array(c['labels'])

    # Compute sign orientation map (same logic best_nadler_on uses internally,
    # repeated here so we can persist the signs alongside the weights).
    sign_map = {}
    for k in keys:
        ap, *_ = boot_auc(labels_arr,  feat_dict[k])
        an, *_ = boot_auc(labels_arr, -feat_dict[k])
        sign_map[k] = +1 if (not np.isnan(ap) and ap >= an) else -1

    auc, lo, hi, subset, weights = best_nadler_on(
        feat_dict, keys, labels_arr,
        max_size=4, label=f'{mk}/{ds}', compare_mean=False,
    )
    NADLER_RES[(mk, ds)] = {
        'auc': auc, 'lo': lo, 'hi': hi,
        'subset': list(subset) if subset else [],
        'weights': list(weights) if weights is not None else [],
        'sign_map': sign_map,
    }

print('=== Best Nadler per cell ===')
for (mk, ds), r in NADLER_RES.items():
    sub_str = ' + '.join(r['subset']) if r['subset'] else '(none)'
    print(f'  [{mk:<10s}/{ds:<22s}] AUC={100*r["auc"]:.1f}% [{100*r["lo"]:.1f},{100*r["hi"]:.1f}]  subset: {sub_str}')

  [qwen7b/hotpotqa] 13 features, 13 informative, max_size=4 → 1079 raw combos
    size=2: 78 combos, 74 passed ρ-filter, best so far=76.3%
    size=3: 286 combos, 243 passed ρ-filter, best so far=79.5%
    size=4: 715 combos, 510 passed ρ-filter, best so far=79.5%
  [qwen7b/hotpotqa] done — checked=827, skipped(ρ)=252, best=79.5%
  [qwen7b/natural_questions] 13 features, 13 informative, max_size=4 → 1079 raw combos
    size=2: 78 combos, 71 passed ρ-filter, best so far=73.2%
    size=3: 286 combos, 213 passed ρ-filter, best so far=75.3%
    size=4: 715 combos, 385 passed ρ-filter, best so far=75.3%
  [qwen7b/natural_questions] done — checked=669, skipped(ρ)=410, best=75.3%
  [qwen7b/2wikimultihopqa] 13 features, 13 informative, max_size=4 → 1079 raw combos
    size=2: 78 combos, 72 passed ρ-filter, best so far=79.5%
    size=3: 286 combos, 224 passed ρ-filter, best so far=79.8%
    size=4: 715 combos, 433 passed ρ-filter, best so far=80.5%
  [qwen7b/2wikimultihopqa] done — checked=729,

In [ ]:
# Cell 15 — Length-controlled analysis per cell
#
# Headline number for the thesis defensibility:
#   spectral-only Nadler AUC vs trace_length-alone AUC.
# If spectral-only consistently beats length-alone, the spectral story holds
# beyond the trivial "longer statements are more grounded" effect.

LEN_RES = {}
for (mk, ds), c in ALL_CELLS.items():
    if not c['features']:
        continue
    keys = list(c['features'][0].keys())
    feat_dict = {k: np.array([f[k] for f in c['features']]) for k in keys}
    labels_arr = np.array(c['labels'])

    # 1) trace_length alone
    tl = feat_dict['trace_length']
    a_tl, lo_tl, hi_tl = boot_auc(labels_arr, tl)
    if not np.isnan(a_tl) and a_tl < 0.5:
        a_tl, lo_tl, hi_tl = boot_auc(labels_arr, -tl)

    # 2) Spectral-only Nadler (excludes trace_length)
    spectral_keys = [k for k in keys if k != 'trace_length']
    a_sp, lo_sp, hi_sp, sub_sp, w_sp = best_nadler_on(
        feat_dict, spectral_keys, labels_arr,
        max_size=4, label=f'{mk}/{ds}-spectralonly', compare_mean=False,
    )

    LEN_RES[(mk, ds)] = {
        'trace_length_auc':   a_tl,
        'trace_length_ci':    (lo_tl, hi_tl),
        'spectral_only_auc':  a_sp,
        'spectral_only_ci':   (lo_sp, hi_sp),
        'spectral_only_subset': list(sub_sp) if sub_sp else [],
        'lift_over_length_pp':  100 * (a_sp - a_tl),
    }

print('=== Length-controlled summary ===')
print(f'{"cell":<35s}  {"len-only":>10s}  {"spec-only":>10s}  {"\u0394 (pp)":>10s}')
for (mk, ds), r in LEN_RES.items():
    print(f'  {mk+"/"+ds:<33s}  {100*r["trace_length_auc"]:>9.1f}%  '
          f'{100*r["spectral_only_auc"]:>9.1f}%  {r["lift_over_length_pp"]:>+9.1f}')

  [qwen7b/hotpotqa-spectralonly] 12 features, 12 informative, max_size=4 → 781 raw combos
    size=2: 66 combos, 63 passed ρ-filter, best so far=76.2%
    size=3: 220 combos, 191 passed ρ-filter, best so far=79.5%
    size=4: 495 combos, 371 passed ρ-filter, best so far=79.5%
  [qwen7b/hotpotqa-spectralonly] done — checked=625, skipped(ρ)=156, best=79.5%
  [qwen7b/natural_questions-spectralonly] 12 features, 12 informative, max_size=4 → 781 raw combos
    size=2: 66 combos, 60 passed ρ-filter, best so far=73.2%
    size=3: 220 combos, 164 passed ρ-filter, best so far=73.7%
    size=4: 495 combos, 270 passed ρ-filter, best so far=74.1%
  [qwen7b/natural_questions-spectralonly] done — checked=494, skipped(ρ)=287, best=74.1%
  [qwen7b/2wikimultihopqa-spectralonly] 12 features, 12 informative, max_size=4 → 781 raw combos
    size=2: 66 combos, 61 passed ρ-filter, best so far=79.1%
    size=3: 220 combos, 174 passed ρ-filter, best so far=79.8%
    size=4: 495 combos, 309 passed ρ-filter, be

In [ ]:
# Cell 16 — PCA diagnostic per cell (PC1 AUC vs Nadler AUC)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

PCA_RES = {}
for (mk, ds), c in ALL_CELLS.items():
    if not c['features']:
        continue
    keys = list(c['features'][0].keys())
    X = np.array([[f[k] for k in keys] for f in c['features']])
    Xs = StandardScaler().fit_transform(X)
    pca = PCA(n_components=min(5, X.shape[1])).fit(Xs)
    pc1 = pca.transform(Xs)[:, 0]
    labels_arr = np.array(c['labels'])
    a_pc1 = roc_auc_score(labels_arr, pc1)
    if a_pc1 < 0.5:
        a_pc1 = roc_auc_score(labels_arr, -pc1)
    loadings = pd.Series(np.abs(pca.components_[0]), index=keys).sort_values(ascending=False)
    PCA_RES[(mk, ds)] = {
        'pc1_auc':         a_pc1,
        'pc1_var_ratio':   float(pca.explained_variance_ratio_[0]),
        'top3_loadings':   loadings.head(3).to_dict(),
        'nadler_lift_over_pc1_pp': 100 * (NADLER_RES[(mk, ds)]['auc'] - a_pc1),
    }

print('=== PCA vs Nadler ===')
print(f'{"cell":<35s}  {"PC1 AUC":>9s}  {"Nadler":>9s}  {"lift":>7s}  PC1 top3')
for (mk, ds), r in PCA_RES.items():
    top3 = ', '.join(f'{k}({v:.2f})' for k, v in r['top3_loadings'].items())
    print(f'  {mk+"/"+ds:<33s}  {100*r["pc1_auc"]:>8.1f}%  '
          f'{100*NADLER_RES[(mk,ds)]["auc"]:>8.1f}%  {r["nadler_lift_over_pc1_pp"]:>+6.1f}  {top3}')

NameError: name 'NADLER_RES' is not defined

## Section 5 — Cross-cell comparisons (the headline visualizations)

These are the figures the user explicitly asked for: how do Nadler weight vectors differ across tasks, and how does the fused score look across cells.

In [ ]:
# Cell 17 — Headline AUC heatmap (4 models × 4 datasets)
import matplotlib.pyplot as plt
import seaborn as sns

model_keys = [m[0] for m in MODELS]
auc_matrix = np.full((len(model_keys), len(DATASETS)), np.nan)
for i, mk in enumerate(model_keys):
    for j, ds in enumerate(DATASETS):
        if (mk, ds) in NADLER_RES:
            auc_matrix[i, j] = NADLER_RES[(mk, ds)]['auc']

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(auc_matrix, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0.5, vmax=0.85,
            xticklabels=DATASETS, yticklabels=model_keys, cbar_kws={'label': 'Nadler AUC'},
            ax=ax, mask=np.isnan(auc_matrix))
ax.set_title('Phase 10 Main \u2014 Nadler AUC across 4 models \u00d7 4 L-CiteEval tasks')
ax.set_xlabel('Dataset (L-CiteEval sub-task)'); ax.set_ylabel('Model')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'A_headline_auc_heatmap.png'), dpi=150)
plt.show()

NameError: name 'NADLER_RES' is not defined

In [ ]:
# Cell 18 — Nadler weight matrix heatmap (16 cells × 13 features)
#
# Each row is one (model, dataset) cell. Each column is one feature.
# Cell value: signed Nadler weight (sign reflects feature orientation).
# Empty cells = feature not in best subset for that (model, dataset).

all_keys = FEAT_COLS or list(next(iter(ALL_CELLS.values()))['features'][0].keys())
row_labels = []
wm = np.zeros((len(NADLER_RES), len(all_keys)))
for i, ((mk, ds), r) in enumerate(NADLER_RES.items()):
    row_labels.append(f'{mk}/{ds}')
    for feat, w in zip(r['subset'], r['weights']):
        j = all_keys.index(feat)
        # apply orientation sign so the weight reflects "how much this feature
        # contributes to the grounded direction\"
        wm[i, j] = w * r['sign_map'][feat]

fig, ax = plt.subplots(figsize=(15, 8))
vmax = max(0.6, np.abs(wm).max())
sns.heatmap(wm, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
            xticklabels=all_keys, yticklabels=row_labels,
            cbar_kws={'label': 'Signed Nadler weight (sign = orientation)'}, ax=ax)
ax.set_title('Nadler weight vectors per (model, dataset) cell \u2014 spectral fingerprint per task')
ax.set_xlabel('Spectral feature'); ax.set_ylabel('Cell')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'B_nadler_weight_matrix.png'), dpi=150)
plt.show()

NameError: name 'NADLER_RES' is not defined

In [ ]:
# Cell 19 — Hierarchical clustering of Nadler weight vectors
#
# Question: do (model, dataset) cells cluster by MODEL or by DATASET?
# - Cluster by dataset \u2192 task type drives the spectral fingerprint
# - Cluster by model  \u2192 model architecture drives it
# - No clear cluster  \u2192 idiosyncratic per cell (less generalizable)

from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist

# Cosine distance on the signed weight vectors (rows of wm)
if wm.shape[0] >= 2:
    Z = linkage(pdist(wm, metric='cosine'), method='average')
    fig, ax = plt.subplots(figsize=(11, 6))
    dendrogram(Z, labels=row_labels, leaf_rotation=45, leaf_font_size=9, ax=ax)
    ax.set_title('Hierarchical clustering of Nadler weight vectors (cosine distance)')
    ax.set_ylabel('Distance')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, 'C_nadler_weight_clustering.png'), dpi=150)
    plt.show()
else:
    print('Need >=2 cells to cluster.')

In [ ]:
# Cell 20 — Fusion-score distribution grid (one panel per cell)
#
# For each (model, dataset), reconstruct the Nadler-fused score per statement
# and overlay grounded vs ungrounded distributions.

n_rows = len(MODELS); n_cols = len(DATASETS)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 3 * n_rows), squeeze=False)

for i, (mk, _, _, _) in enumerate(MODELS):
    for j, ds in enumerate(DATASETS):
        ax = axes[i, j]
        if (mk, ds) not in ALL_CELLS or not NADLER_RES.get((mk, ds), {}).get('subset'):
            ax.text(0.5, 0.5, 'no data', ha='center', va='center')
            ax.set_title(f'{mk} / {ds}', fontsize=9)
            continue
        c = ALL_CELLS[(mk, ds)]
        keys = list(c['features'][0].keys())
        feat_dict = {k: np.array([f[k] for f in c['features']]) for k in keys}
        labels_arr = np.array(c['labels'])
        r = NADLER_RES[(mk, ds)]
        oriented = [zscore(feat_dict[k] * r['sign_map'][k]) for k in r['subset']]
        fused, _ = nadler_fuse(*oriented)
        for cls, color, lbl in [(1, 'steelblue', 'grounded'), (0, 'firebrick', 'ungrounded')]:
            mask = labels_arr == cls
            if mask.sum() > 1:
                ax.hist(fused[mask], bins=18, alpha=0.55, color=color, label=lbl, density=True)
        ax.set_title(f'{mk}/{ds}  AUC={100*r["auc"]:.1f}%', fontsize=9)
        ax.tick_params(labelsize=8)
        if i == 0 and j == 0:
            ax.legend(fontsize=8)

fig.suptitle('Nadler-fused score distributions per (model, dataset) cell', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'D_fusion_distributions_grid.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 21 \u2014 Length-controlled side-by-side bar chart
import matplotlib.patches as mpatches

ordered = list(LEN_RES.items())
ordered.sort(key=lambda x: (x[0][0], x[0][1]))
labels = [f'{mk}\n{ds}' for (mk, ds), _ in ordered]
len_aucs = [100*r['trace_length_auc']  for _, r in ordered]
spec_aucs = [100*r['spectral_only_auc'] for _, r in ordered]
full_aucs = [100*NADLER_RES[(mk, ds)]['auc'] for (mk, ds), _ in ordered]

x = np.arange(len(labels)); w = 0.27
fig, ax = plt.subplots(figsize=(15, 5))
ax.bar(x - w, len_aucs,  w, label='trace_length alone', color='gray',     alpha=0.85)
ax.bar(x,     spec_aucs, w, label='Spectral-only Nadler', color='steelblue', alpha=0.85)
ax.bar(x + w, full_aucs, w, label='Full Nadler (with length)', color='seagreen', alpha=0.85)
ax.axhline(50, color='black', linestyle=':', linewidth=0.8)
ax.axhline(70, color='gold',  linestyle='--', linewidth=0.8, label='G1 target (70%)')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('AUC (%)')
ax.set_title('Length-controlled comparison: Spectral-only Nadler vs trace_length-alone vs Full Nadler')
ax.legend(loc='upper right', fontsize=8)
ax.set_ylim(45, 95)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'E_length_controlled_bars.png'), dpi=150)
plt.show()

## Section 6 \u2014 Baselines

In [ ]:
# Cell 22 — Lite Semantic Entropy baseline (k=10 MC samples per statement)
#
# Cost: SE_SAMPLES × ~3 statements × SE_K = 1500 generations per cell × 16 cells.
# Implemented as Jaccard-clustered SE (no NLI dependency — see baselines.py).
# Per-cell budget: ~2 hours on Qwen-72B-AWQ; faster on smaller models.

SE_DIR = os.path.join(BASE_DIR, 'se_baseline')
os.makedirs(SE_DIR, exist_ok=True)

def se_path(model_key, ds_name):
    return os.path.join(SE_DIR, f'se_{model_key}__{ds_name}.pkl')

def _stub_pcre_with_re():
    """Stub gptqmodel's pcre import with stdlib re — see Cell 9 for rationale."""
    if 'pcre' in sys.modules and getattr(sys.modules['pcre'], '__stubbed__', False):
        return
    import re as _re, types as _types
    _pcre = _types.ModuleType('pcre')
    _pcre.__stubbed__ = True
    for _fn in ('compile', 'match', 'search', 'findall', 'sub', 'split', 'fullmatch'):
        setattr(_pcre, _fn, getattr(_re, _fn))
    _pcre.Pattern = _re.Pattern
    _pcre.Match   = _re.Match
    _pcre.error   = _re.error
    _flag_names = ('IGNORECASE', 'MULTILINE', 'DOTALL', 'VERBOSE', 'UNICODE', 'ASCII')
    for _flag in _flag_names:
        setattr(_pcre, _flag, getattr(_re, _flag))
    _pcre.Flag = _types.SimpleNamespace(**{_f: getattr(_re, _f) for _f in _flag_names})
    sys.modules['pcre'] = _pcre

def _install_gptqmodel_runtime_deps():
    """gptqmodel runtime deps that `--no-deps` skips. See Cell 9 for rationale."""
    os.system('pip install -q --no-deps device-smi tokenicer defuser')
    os.system('pip install -q logbar')
    os.system('pip install -q --no-deps gptqmodel')

def run_se_for_model(model_key, model_id, model_kwargs):
    needed = [ds for ds in DATASETS if (model_key, ds) in ALL_CELLS
                                       and not os.path.exists(se_path(model_key, ds))]
    if not needed:
        print(f'  [{model_key}] all SE cells already done')
        return
    if 'awq' in model_id.lower() or 'gptq' in model_id.lower():
        _stub_pcre_with_re()
        _install_gptqmodel_runtime_deps()
    if 'meta-llama/Llama-3.3-70B' in model_id:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        tok = AutoTokenizer.from_pretrained(model_id, token=hf_token)
        if tok.pad_token is None: tok.pad_token = tok.eos_token
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                                 bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4')
        mdl = AutoModelForCausalLM.from_pretrained(
            model_id, device_map={'': 0}, quantization_config=bnb,
            attn_implementation='eager', trust_remote_code=False, token=hf_token).eval()
    else:
        mdl, tok = load_model(model_id, **model_kwargs)

    for ds in needed:
        c = ALL_CELLS[(model_key, ds)]
        statements = c['statements'][:SE_SAMPLES]
        labels     = c['labels'][:SE_SAMPLES]
        se_scores = []
        for stmt in tqdm(statements, desc=f'SE {model_key}/{ds}'):
            full_text = stmt['full_text']
            ts        = stmt['token_start_in_response']
            re_enc = tok(full_text, return_offsets_mapping=True, add_special_tokens=False)
            offs   = re_enc['offset_mapping']
            char_cut = offs[ts][0] if ts < len(offs) else len(full_text)
            prompt_prefix = lciteeval_prompt(stmt['row']) + '\n' + full_text[:char_cut]
            try:
                se = lite_semantic_entropy_for_statement(
                    mdl, tok, prompt_prefix,
                    statement_token_start_global=0,
                    n_continuation_tokens=32, k=SE_K, temperature=TEMP, seed=SEED,
                )
            except Exception as ex:
                print(f'    SE error: {ex}')
                se = float('nan')
            se_scores.append(se)
        with open(se_path(model_key, ds), 'wb') as f:
            pickle.dump({'labels': labels, 'se_scores': se_scores}, f)
        print(f'  [{model_key}/{ds}] SE done ({len(se_scores)} statements)')
    del mdl, tok; free_memory()

# To run, uncomment the model(s) you want to score.
# for mk, mid, kw, _ in MODELS:
#     run_se_for_model(mk, mid, kw)
print('SE baseline scaffolded — uncomment the loop above to run.')

In [ ]:
# Cell 23 \u2014 Aggregate SE results vs Nadler results
SE_RES = {}
for mk, _, _, _ in [m for m in MODELS]:
    for ds in DATASETS:
        p = se_path(mk, ds)
        if not os.path.exists(p):
            continue
        d = pickle.load(open(p, 'rb'))
        labs = np.array(d['labels'])
        scs  = np.array(d['se_scores'])
        valid = ~np.isnan(scs)
        if valid.sum() < 10 or len(np.unique(labs[valid])) < 2:
            continue
        a, lo, hi = boot_auc(labs[valid], scs[valid])
        if not np.isnan(a) and a < 0.5:
            a, lo, hi = boot_auc(labs[valid], -scs[valid])
        SE_RES[(mk, ds)] = {'auc': a, 'lo': lo, 'hi': hi, 'n': int(valid.sum())}

if SE_RES:
    print(f'{"cell":<35s}  {"SE AUC":>10s}  {"Nadler":>10s}  {"\u0394 (pp)":>10s}  {"n_se":>5s}')
    for (mk, ds), s in SE_RES.items():
        n = NADLER_RES.get((mk, ds), {}).get('auc', float('nan'))
        delta = 100 * (n - s['auc']) if not np.isnan(n) else float('nan')
        print(f'  {mk+"/"+ds:<33s}  {100*s["auc"]:>9.1f}%  {100*n:>9.1f}%  {delta:>+9.1f}  {s["n"]:>5d}')
else:
    print('No SE results yet. Run Cell 22 first.')

## Section 7 \u2014 Decision gates + final summary

In [ ]:
# Cell 24 \u2014 Per-cell gate evaluation + cross-cell aggregate
#
# G1: Nadler AUC \u2265 70% on \u22656/16 cells
# G2: Nadler beats SE by \u22653pp on \u22656/16 cells (only counts cells with SE computed)
# G3: Spectral-only Nadler beats trace_length-alone by \u22653pp on \u22658/16 cells (length-confound control)
# G4: Cross-family transfer \u2014 both Qwen-72B and Llama-70B \u2265 65% on \u22653/4 datasets

gate_rows = []
for (mk, ds), r in NADLER_RES.items():
    se = SE_RES.get((mk, ds), {}).get('auc', float('nan'))
    lr = LEN_RES.get((mk, ds), {})
    gate_rows.append({
        'cell':                  f'{mk}/{ds}',
        'nadler_auc':            r['auc'],
        'spec_only_auc':         lr.get('spectral_only_auc', float('nan')),
        'trace_length_auc':      lr.get('trace_length_auc', float('nan')),
        'se_auc':                se,
        'G1 (\u226570%)':            '\u2713' if r['auc'] >= 0.70 else '\u2717',
        'G2 (Nadler-SE\u22653pp)':   ('\u2713' if (not np.isnan(se) and r['auc'] - se >= 0.03)
                                  else ('-' if np.isnan(se) else '\u2717')),
        'G3 (spec-len\u22653pp)':    ('\u2713' if (lr and lr['spectral_only_auc'] - lr['trace_length_auc'] >= 0.03)
                                  else '\u2717'),
    })
df_gates = pd.DataFrame(gate_rows)
print('=== Per-cell gates ===')
print(df_gates.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

# Cross-cell aggregate
n_cells = len(NADLER_RES)
n_g1    = sum(1 for r in NADLER_RES.values() if r['auc'] >= 0.70)
n_g2    = sum(1 for (k,v) in NADLER_RES.items()
              if k in SE_RES and v['auc'] - SE_RES[k]['auc'] >= 0.03)
n_g3    = sum(1 for r in LEN_RES.values()
              if r['spectral_only_auc'] - r['trace_length_auc'] >= 0.03)

# G4: cross-family transfer
g4_passes = 0
for ds in DATASETS:
    qwen72 = NADLER_RES.get(('qwen72b',  ds), {}).get('auc', 0)
    llama  = NADLER_RES.get(('llama70b', ds), {}).get('auc', 0)
    if qwen72 >= 0.65 and llama >= 0.65:
        g4_passes += 1

print('\n=== Cross-cell aggregate ===')
print(f'G1 \u2014 Nadler \u2265 70%      : {n_g1}/{n_cells} cells  (target: \u22656)')
print(f'G2 \u2014 Nadler beats SE by \u22653pp : {n_g2}/{len(SE_RES)} SE cells  (target: \u22656)')
print(f'G3 \u2014 Spec-only beats len-alone by \u22653pp : {n_g3}/{len(LEN_RES)} cells  (target: \u22658)')
print(f'G4 \u2014 Both 70B models \u2265 65%   : {g4_passes}/4 datasets  (target: \u22653)')

verdict_parts = []
if n_g1 >= 6: verdict_parts.append('G1\u2713')
else:         verdict_parts.append('G1\u2717')
if n_g2 >= 6: verdict_parts.append('G2\u2713')
else:         verdict_parts.append('G2\u2717')
if n_g3 >= 8: verdict_parts.append('G3\u2713')
else:         verdict_parts.append('G3\u2717')
if g4_passes >= 3: verdict_parts.append('G4\u2713')
else:              verdict_parts.append('G4\u2717')
print(f'\nVerdict: {" ".join(verdict_parts)}')

In [ ]:
# Cell 25 \u2014 Save the comprehensive results pickle
FINAL = {
    'config':         {'models': MODELS, 'datasets': DATASETS,
                       'n_samples': N_SAMPLES, 'temp': TEMP, 'max_new': MAX_NEW,
                       'se_samples': SE_SAMPLES, 'se_k': SE_K},
    'nadler':         NADLER_RES,
    'length_control': LEN_RES,
    'pca':            PCA_RES,
    'semantic_entropy': SE_RES,
    'gates':          {'g1_n': n_g1, 'g2_n': n_g2, 'g3_n': n_g3, 'g4_n': g4_passes,
                       'n_cells': n_cells},
    'feat_keys':      FEAT_COLS,
}
with open(os.path.join(RES_DIR, 'phase10_main_results.pkl'), 'wb') as f:
    pickle.dump(FINAL, f)
print(f'Saved: {os.path.join(RES_DIR, "phase10_main_results.pkl")}')
print(f'\nPlot files in: {PLOT_DIR}')

## Section 8 \u2014 Comparison to literature + thesis framing

### RAG baselines (target table)

From `research_phase10_rag/`:

| Method | Reported | Where | Our headline |
|---|---|---|---|
| **TOHA** (topological divergence on attention graphs) | HotpotQA AUC 0.71\u20130.80 | Bazarova et al. 2025, ACL 2026 | (compare to our hotpotqa cells) |
| **Semantic Entropy** (Kuhn et al. 2023) | ~0.71 long-form | Real-Time Probes paper | (computed in Cell 23) |
| **Real-Time Entity Probes** (linear probes on hidden states) | LongFact AUC 0.90 | Obeso et al. 2025 | complementary signal (entity-level vs trajectory-level) |
| **LapEigvals** (top-k Laplacian eigenvalues of attention) | TriviaQA 0.83\u20130.89 | EMNLP 2025 | different axis (static attention vs entropy time series) |
| **L-CiteEval native CR/CP/F1** | task quality, not detection | Tang et al. 2024, ACL 2025 | not directly comparable |

### Position vs concurrent spectral work

- **Energy Mountain (arXiv:2510.19117)**: spectral features on the **layer / depth axis**. We are on the **generation-time axis**. Different signal, different graph, different axis.
- **Streaming Prefix-Level (arXiv:2601.02170)**: aggregates entropy as a scalar; no FFT/Fourier/eigenvalue analysis. Different signal.
- **SIA (arXiv:2604.06192)**: theoretically names entropy-spike-as-failure-signal as a corollary. **Strengthens** our framing.

### Connection to thesis goals (RAG + Agentic chapter)

This notebook establishes **Domain 1 \u2014 RAG**: long-doc grounded generation with citation-level labels. The complementary **Domain 2 \u2014 Agentic** (multi-step trajectories with Thought/Action/Observation, e.g. DeepHalluBench) requires a Deep Research Agent harness and is the subject of `Spectral_Analysis_Phase10_Agentic.ipynb` (future).

Together the two notebooks form the Phase 10 chapter: **\"the spectral pipeline extends from math reasoning into long-form grounded and agentic generation, wherever the model produces structured multi-step output\"** \u2014 a coherent thesis claim that unifies the RAG and Agentic directions of `Research_Directions.md` rather than splitting them into two separate chapters.

### Supervisor connections

- **Bracha (LTT/conformal)**: `Spectral_Analysis_Phase10_Conformal.ipynb` will take the validated Nadler scores from this notebook and produce a calibrated detector with FNR \u2264 \u03b1 guarantee.
- **Ofir (multi-view kernel/anomaly)**: the Nadler weight vectors per cell (Cell 18) are exactly the kind of multi-view fingerprint his framework was designed to fuse. A natural follow-up: VSDE-style anomaly detection on the spectral feature space, compared against Nadler.

### Next concrete steps

1. After this notebook completes: append HISTORY.md Step 85 with the headline numbers.
2. Update `Research_Directions.md` Direction 2 (RAG) status.
3. Start `Spectral_Analysis_Phase10_Conformal.ipynb` (Bracha layer).
4. Scope `Spectral_Analysis_Phase10_Agentic.ipynb` based on which open-source DRA harness is feasible (Qwen DRA is the leading candidate per `deephallu_bench.json`).
